# Demo 5 — Carrying the overhead: window & cost
<a id="top"></a>

```yaml
title: Carrying the overhead — window and cost
keywords: context window, cost, pricing, input output asymmetry, staircase, budgeting
estimated duration: 15
```

> **Lesson L01**, teacher demo 5 — the payoff. Written lecture [L0102_lecture.md](L0102_lecture.md) §7–8.
>
> Fully **offline**: token counts via `tiktoken`, costs via **illustrative** rates. Because you
> re-send the preamble + history every call (no memory), those tokens hit two budgets at once —
> **space** (the window) and **money** (cost).

**Contents**

1. [The window is one shared budget](#1-the-window-is-one-shared-budget)
2. [Three failure modes](#2-three-failure-modes)
3. [Cost: per token, both directions](#3-cost-per-token-both-directions)
4. [The staircase and orders of magnitude](#4-the-staircase-and-orders-of-magnitude)

In [1]:
from __future__ import annotations

import tiktoken

ENC = tiktoken.get_encoding("cl100k_base")
WINDOW = 200_000  # Claude Sonnet 4.6 standard context window (tokens)

# ⚠️ ILLUSTRATIVE rates only ($ per 1,000,000 tokens). Pricing changes — pull current Claude
# Sonnet numbers from Anthropic's pricing page before quoting any real figure.
INPUT_RATE = 3.0
OUTPUT_RATE = 15.0


def window_meter(segments: dict[str, int], total: int = WINDOW, width: int = 40) -> None:
    used = sum(segments.values())
    filled = round(width * min(used, total) / total)
    print(f"[{'#' * filled}{'-' * (width - filled)}] {used:,} / {total:,} tokens")
    for name, n in segments.items():
        print(f"    {name:16s} {n:>8,}")


def call_cost_usd(input_tokens: int, output_tokens: int) -> float:
    return input_tokens / 1e6 * INPUT_RATE + output_tokens / 1e6 * OUTPUT_RATE

## 1. The window is one shared budget

The preamble, every prior turn, the current input, and the reserved output all draw from the
**same** 200k window. Watch it fill as history grows.

In [2]:
window_meter({"preamble": 300, "history": 1_200, "current input": 250, "reserved output": 1_024})
print()
# ...now the same call after a long conversation:
window_meter({"preamble": 300, "history": 160_000, "current input": 250, "reserved output": 1_024})

[#---------------------------------------] 2,774 / 200,000 tokens
    preamble              300
    history             1,200
    current input         250
    reserved output     1,024

[################################--------] 161,574 / 200,000 tokens
    preamble              300
    history           160,000
    current input         250
    reserved output     1,024


## 2. Three failure modes

| Failure mode        | What happens                                          | Danger |
| ------------------- | ----------------------------------------------------- | ------ |
| Hard rejection      | API refuses the request before running it             | Loud — easy to catch |
| Silent truncation   | Some frameworks quietly drop the oldest turns         | **Quiet — the call succeeds but the model "forgot"** |
| Quality degradation | Even when it fits, mid-context material gets lost      | Subtle — vaguer answers, no error |

The dangerous one is **silent truncation**: no error, but the model no longer has context you
assumed. It motivates context management (L19) and RAG (L21).

[↑ Back to top](#top)

## 3. Cost: per token, both directions

Separate input and output rates, on every call. Output usually costs **3–5× more** — so a long
prompt + short answer often beats a short prompt + long answer.

In [3]:
print(f"2000 in / 50 out : ${call_cost_usd(2000, 50):.5f}")
print(f"  50 in / 2000 out: ${call_cost_usd(50, 2000):.5f}   <- output is the expensive direction")

2000 in / 50 out : $0.00675
  50 in / 2000 out: $0.03015   <- output is the expensive direction


## 4. The staircase and orders of magnitude

No memory means every turn re-sends the whole history **plus the preamble** — so input tokens
climb even when each message is small.

In [4]:
cumulative = 0
session_cost = 0.0
for turn in range(1, 6):
    cumulative += 200  # ~200 new tokens/turn, but ALL prior tokens are re-sent
    cost = call_cost_usd(cumulative, 60)
    session_cost += cost
    print(f"turn {turn}: input re-sent={cumulative:>4} tokens  call=${cost:.5f}")
print(f"\n5-turn session ~ ${session_cost:.4f}")

one_call = call_cost_usd(2000, 500)
print("\norder-of-magnitude ladder (illustrative):")
for label, mult in [
    ("1 call", 1),
    ("x10 agent run", 10),
    ("x100 dev loop", 100),
    ("x1000 deploy", 1000),
]:
    print(f"  {label:16s} ~ ${one_call * mult:.2f}")

turn 1: input re-sent= 200 tokens  call=$0.00150
turn 2: input re-sent= 400 tokens  call=$0.00210
turn 3: input re-sent= 600 tokens  call=$0.00270
turn 4: input re-sent= 800 tokens  call=$0.00330
turn 5: input re-sent=1000 tokens  call=$0.00390

5-turn session ~ $0.0135

order-of-magnitude ladder (illustrative):
  1 call           ~ $0.01
  x10 agent run    ~ $0.14
  x100 dev loop    ~ $1.35
  x1000 deploy     ~ $13.50


⚠️ **Pricing note:** the rates here are illustrative. Per-token pricing changes — always pull the
current Claude Sonnet numbers from Anthropic's pricing page before quoting a real figure.

The bow: everything you front-loaded to predict better is overhead you re-send, re-count, and
re-pay on every call — against both budgets. *Every prompt is a budget decision.*

[↑ Back to top](#top)